Documentation: [readme.md](readme.md). The first code cell switches the process **cwd** to **`deploy/`** so `utils.py` imports work whether you start Jupyter from the **repo root** or from **`deploy/`**.

Environment: from the **repository root**, run **`uv sync`** (creates **`.venv/`** at the repo root) then **`uv run jupyter notebook deploy/deploy.ipynb`** (see readme).

**Firmware `.bin`**: store under **`deploy/bin/`** or rely on auto-download inside **`flash_with_esptool`** / **`run_full_flash`** (see readme).

Deployments are **Step 1** (serial flash) and **Step 2** (copy or update firmware on `CIRCUITPY`). Step 2 **chooses automatically**: if **`boot.toml`** already exists on the volume, it runs the **update** path (settings backup + merge + restore); otherwise it runs the **new-board** copy. **Run all** runs Step 1 then Step 2 — only run Step 1 when you need a full CircuitPython reflash.


## Configuration

Run the **setup** cell below, then **Serial port** to list USB devices and optionally pick one for **`CFG.serial_port`** (default **`/dev/tty.usbmodem101`**).

You can also edit **`CFG`** directly: `circuitpy_root`, and optional **`circuitpython_bin`** / **`circuitpython_board_id`** / **`circuitpython_locale`** / **`circuitpython_download_fallback_url`**`. Settings backups use **`device_id`** from the device `settings.toml` under **`deploy/settings_backups/<device_id>/`**.

In [ ]:
import sys
from pathlib import Path

# So ``import notebook_env`` works from repo root or from deploy/
_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import (
    DeployConfig,
    copy_firmware_to_circuitpy,
    flash_with_esptool,
    list_serial_ports,
    pick_serial_port_interactive,
    run_full_flash,
    run_update_only,
)

# Override defaults, e.g. DeployConfig(circuitpy_root=Path("/media/youruser/CIRCUITPY"))  # Linux CIRCUITPY path
CFG = DeployConfig()
CFG

### Serial port

Run the next cell to **probe USB serial ports** (requires `pyserial`). It **bootstraps `deploy/`** like the setup cell, so it still works after a **kernel restart** without re-running setup. Type a **number** and Enter to set **`CFG.serial_port`**, or **Enter alone** to keep the current value (default **`/dev/tty.usbmodem101`**). If **`CFG`** does not exist yet, the cell creates it.

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import DeployConfig, pick_serial_port_interactive

if "CFG" not in globals():
    CFG = DeployConfig()

pick_serial_port_interactive(CFG)
CFG

## Step 1 — Flash the board

Serial flashing only (**`erase_flash`** if enabled, then **`write_flash`**). Resolves a **`.bin`** via **`ensure_circuitpython_bin`** (existing file, `deploy/bin/`, download index, or fallback). Does **not** copy the repo onto **`CIRCUITPY`**.

Use **BOOT + RESET** on ESP32-S3 if the port does not appear. After this step, reset the board and wait until **`CIRCUITPY`** mounts before Step 2 (copy or update).

In [ ]:
flash_with_esptool(CFG)

## Step 2 — Copy or update firmware on CIRCUITPY

Run the code cell below after **`CIRCUITPY`** is mounted (reset the board if needed).

- **If `boot.toml` exists** on `CFG.circuitpy_root` → **update** (existing board): same as the former Step 3 — back up `settings.toml` / `startup.toml`, copy the repo **`firmware/`** tree, merge any **new keys** from repo templates into the slot backup, restore settings to the board.
- **If `boot.toml` is missing** → **new board** (same as the former Step 2): copy the **`firmware/`** tree plus **`deploy/settings.toml`** from this repo.

While the **`firmware/`** tree is copied, output shows a **tqdm** progress bar (install deps with **`uv sync`** at the repo root if the bar does not appear).


In [ ]:
_boot = CFG.circuitpy_root / "boot.toml"
if _boot.is_file():
    print(f"Found {_boot} — updating firmware (existing board).", flush=True)
    run_update_only(CFG)
else:
    print(f"No {_boot} — copying firmware tree (new board).", flush=True)
    copy_firmware_to_circuitpy(CFG)


## Optional — List serial ports

Requires **`pyserial`**. Use the printed device path for **`CFG.serial_port`**.

In [ ]:
list_serial_ports()

## Optional — Shortcut for a new board

**`run_full_flash(CFG)`** runs **Step 1** then **`copy_firmware_to_circuitpy`** (equivalent to the **no `boot.toml`** branch of Step 2). Use this **instead** of running the Step 1 and Step 2 cells above; leave the next line commented if you already use those cells, or you will flash twice.


In [ ]:
# run_full_flash(CFG)  # uncomment only if you skip Step 1 and Step 2 cells above